<a href="https://colab.research.google.com/github/patriciarrs/Rag-finalproject/blob/main/v3/RAG_FinalProject.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 1. Final Project Information - RAG [don't edit this section]

## IMPORTANT
**Add comments to your code whenever necessary to explain your logic.**

## Submission
- Single Deliverable: Google Colab Notebook
- File Name Format: [YourName]_RAG_FinalProject.ipynb (optional but it helps us track your file)
- Submit as: ZIP file containing only the .ipynb file (the student portal doesn't accept .ipynb files)

## Mandatory
- Ingestion Phase
- Inference Phase

# 2. Project Specific Information

## Project Description
This RAG system allows users to ask questions about the Bitcoin and Ethereum whitepapers.
It loads the PDFs, chunks and embeds the text into a vector store, and answers questions
by retrieving the most relevant passages and generating a response with an LLM.
v2 adds a Gradio chat interface so users can interact with the system without writing any code.
v3 adds RAGAS evaluation to measure retrieval and answer quality with objective metrics.

## Steps
1. Load Bitcoin and Ethereum whitepapers (PDFs)
2. Clean and preprocess the text
3. Split documents into chunks
4. Embed chunks and store in Pinecone
5. Build a retrieval chain with a prompt template and an LLM
6. Support multi-turn conversations via chat history
7. Launch a Gradio chat interface
8. Test with example queries
9. Evaluate with RAGAS (Faithfulness, Answer Relevancy, Context Precision, Context Recall)

## Features Implemented
- [x] RAG v2 baseline (mandatory)
- [x] Text preprocessing (clean PDF noise)
- [x] Auto topic detection via LLM
- [x] Chat history support (last 5 turns)
- [x] Gradio chat interface
- [x] RAGAS evaluation

# 3. Installation

In [1]:
%pip install "numpy<2.0" -qqq

In [2]:
# Install all required packages
%pip install "langchain==0.3.27" -qqq
%pip install "langchain-community==0.3.31" -qqq
%pip install "langchain-openai==0.3.35" -qqq
%pip install "langchain-pinecone==0.2.0" -qqq
%pip install "pinecone==5.4.2" -qqq
%pip install pypdf -qqq
%pip install gradio -qqq
#──────────────────────────────────────────────
# NEW in v3: RAGAS evaluation framework
#──────────────────────────────────────────────
%pip install ragas -qqq
%pip install datasets -qqq

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
pinecone 5.4.2 requires pinecone-plugin-inference<4.0.0,>=2.0.0, but you have pinecone-plugin-inference 1.1.0 which is incompatible.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
pinecone-client 5.0.1 requires pinecone-plugin-inference<2.0.0,>=1.0.3, but you have pinecone-plugin-inference 3.1.0 which is incompatible.


# 4. Setup

In [3]:
# ─────────────────────────────────────────────────────────────
# load_secrets() works in two environments:
#   - Google Colab  → reads from Colab Secrets (sidebar 🔑)
#   - Local machine → reads from a .env file in the project root
# ─────────────────────────────────────────────────────────────
def load_secrets(keys):
    import os

    try:
        # Try Colab first
        from google.colab import userdata
        values = {key: userdata.get(key) for key in keys}
        in_colab = True
    except Exception:
        # Fall back to .env file for local development
        try:
            from dotenv import load_dotenv
            load_dotenv()
        except ImportError:
            raise ImportError(
                "python-dotenv is required for local development. "
                "Install it with: pip install python-dotenv"
            )
        values = {key: os.getenv(key) for key in keys}
        in_colab = False

    # Check that all required keys are present
    missing = [key for key, value in values.items() if not value]
    if missing:
        if in_colab:
            raise ValueError(f"Missing keys: {', '.join(missing)}. Set them in Colab Secrets.")
        else:
            raise ValueError(
                f"Missing keys: {', '.join(missing)}.\n\n"
                "Create a .env file in the project root with:\n"
                + "\n".join([f"{key}=YOUR_VALUE_HERE" for key in missing])
            )

    # Expose all keys as environment variables so libraries can pick them up automatically
    for key, value in values.items():
        os.environ[key] = value

    return values

In [4]:
# Load API keys
# Make sure PINECONE_API_KEY and OPENAI_API_KEY
# are set in Colab Secrets (or your local .env file)
try:
    secrets = load_secrets(['PINECONE_API_KEY', 'OPENAI_API_KEY'])
except ValueError as e:
    print(e)

# 5. Global Variables

In [5]:
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_pinecone import PineconeVectorStore
from pinecone import Pinecone, ServerlessSpec
from langchain.text_splitter import RecursiveCharacterTextSplitter

# ── Models ────────────────────────────────────────────────────
# Embeddings: converts text chunks into vector representations
# text-embedding-3-small produces 1536-dimensional vectors
embeddings = OpenAIEmbeddings(
    model="text-embedding-3-small"
)

# Main LLM: generates answers based on retrieved context
llm = ChatOpenAI(
    model="gpt-4o-mini"
)

# Lightweight LLM for classification tasks (cheaper + faster)
classification_llm = ChatOpenAI(
    model="gpt-3.5-turbo",
    temperature=0   # temperature=0 → deterministic output (good for classification)
)

# ── Pinecone Setup ────────────────────────────────────────────
# Pinecone is a managed cloud vector database.
# We create the index once; if it already exists, this is skipped.
INDEX_NAME = "crypto-whitepapers"
EMBEDDING_DIMENSIONS = 1536   # Must match text-embedding-3-small output size

# Initialize Pinecone client with our API key
pc = Pinecone(api_key=secrets["PINECONE_API_KEY"])

# Create the index if it doesn't exist yet
if INDEX_NAME not in pc.list_indexes().names():
    pc.create_index(
        name=INDEX_NAME,
        dimension=EMBEDDING_DIMENSIONS,
        metric="cosine",              # Cosine similarity is standard for text embeddings
        spec=ServerlessSpec(
            cloud="aws",              # Pinecone free tier runs on AWS
            region="us-east-1"
        )
    )
    print(f"✓ Created Pinecone index: '{INDEX_NAME}'")
else:
    print(f"✓ Pinecone index '{INDEX_NAME}' already exists")

# Connect LangChain to our Pinecone index
vectorstore = PineconeVectorStore(
    index_name=INDEX_NAME,
    embedding=embeddings,
    pinecone_api_key=secrets["PINECONE_API_KEY"]
)

# ── Text Splitter ─────────────────────────────────────────────
# Splits large documents into smaller overlapping chunks.
# chunk_size=1000: ~1000 characters per chunk (fits well in LLM context)
# chunk_overlap=200: 20% overlap so sentences aren't cut off between chunks
splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)

✓ Created Pinecone index: 'crypto-whitepapers'


# 6. Indexing / Ingestion Phase

In [6]:
import re

def clean_text(text: str) -> str:
    """
    TEXT PREPROCESSING - Remove noise from raw PDF text.

    PDFs often contain extra whitespace, page numbers, and
    other artifacts that hurt retrieval quality if left in.

    Args:
        text (str): Raw text extracted from a PDF page

    Returns:
        str: Cleaned text
    """
    # Collapse multiple spaces/newlines into a single space
    text = re.sub(r'\s+', ' ', text)

    # Remove standalone page numbers that appear at the end of lines
    text = re.sub(r'(?<=\.)\s*\d+\s*$', '', text)

    # Strip leading/trailing whitespace
    text = text.strip()

    return text

In [7]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

def detect_document_topic(documents: list) -> str:
    """
    AUTO-DETECT TOPIC - Use an LLM to figure out what the document is about.

    We sample the first 3 pages and ask the LLM for a one-word topic.
    This gets stored as metadata on each chunk so we can filter later.

    Args:
        documents (list): List of Document objects loaded from a PDF

    Returns:
        str: Detected topic (e.g., 'bitcoin', 'ethereum')
    """

    # Prompt that asks for a single-word topic label
    topic_detection_template = ChatPromptTemplate.from_template(
        """
        Analyze the following document content and determine its primary topic.

        Document content:
        {content}

        Based on this content, what is the primary topic?
        Answer with a single word or short phrase (e.g., 'bitcoin', 'ethereum').

        Primary topic:
        """
    )

    # Chain: prompt → classification LLM → plain string output
    topic_detection_chain = topic_detection_template | classification_llm | StrOutputParser()

    # Sample only the first 3 pages to keep costs low
    sample_content = ""
    for doc in documents[:3]:
        sample_content += doc.page_content + "\n\n"
    sample_content = sample_content[:4000]  # Cap at 4000 chars

    detected_topic = topic_detection_chain.invoke({"content": sample_content}).strip().lower()

    return detected_topic

In [8]:
from langchain.document_loaders import PyPDFLoader

def ingest_documents(document_path: str):
    """
    INGESTION PIPELINE - Load, clean, chunk, and store a PDF.

    Steps:
      1. Load PDF pages
      2. Auto-detect topic
      3. Clean text
      4. Add metadata
      5. Split into chunks
      6. Store in Pinecone

    Args:
        document_path (str): Local file path or URL to a PDF
    """

    print("-" * 60)
    print(f"INGESTING: {document_path}")
    print("-" * 60)

    # Step 1: Load - PyPDFLoader extracts text page by page
    print("\n[1/5] Loading PDF...")
    loader = PyPDFLoader(document_path)
    documents = loader.load()
    print(f"  ✓ Loaded {len(documents)} pages")

    # Step 2: Auto-detect topic using LLM
    print("\n[2/5] Detecting topic...")
    topic = detect_document_topic(documents)
    print(f"  ✓ Topic: '{topic}'")

    # Step 3: Clean text on every page
    print("\n[3/5] Cleaning text...")
    for doc in documents:
        doc.page_content = clean_text(doc.page_content)
    print(f"  ✓ Cleaned {len(documents)} pages")

    # Step 4: Add topic as metadata so we can filter by it later
    print("\n[4/5] Adding metadata...")
    for doc in documents:
        doc.metadata["topic"] = topic
    print(f"  ✓ Metadata added")

    # Step 5: Split pages into smaller overlapping chunks
    print("\n[5/5] Chunking and storing...")
    chunks = splitter.split_documents(documents)

    # Step 6: Store chunks in Pinecone (embeddings are computed automatically)
    vectorstore.add_documents(chunks)
    print(f"  ✓ Stored {len(chunks)} chunks in Pinecone")

    print(f"\n✓ Done! '{topic}' ingested successfully.\n")

In [9]:
# Ingest both whitepapers
# The Bitcoin whitepaper is publicly available as a URL.
# Download the Ethereum PDF and place it in the same folder as this notebook.
ingest_documents("https://bitcoin.org/bitcoin.pdf")
ingest_documents("./ethereum.pdf")

------------------------------------------------------------
INGESTING: https://bitcoin.org/bitcoin.pdf
------------------------------------------------------------

[1/5] Loading PDF...
  ✓ Loaded 9 pages

[2/5] Detecting topic...
  ✓ Topic: 'bitcoin'

[3/5] Cleaning text...
  ✓ Cleaned 9 pages

[4/5] Adding metadata...
  ✓ Metadata added

[5/5] Chunking and storing...
  ✓ Stored 29 chunks in Pinecone

✓ Done! 'bitcoin' ingested successfully.

------------------------------------------------------------
INGESTING: ./ethereum.pdf
------------------------------------------------------------

[1/5] Loading PDF...
  ✓ Loaded 36 pages

[2/5] Detecting topic...
  ✓ Topic: 'ethereum'

[3/5] Cleaning text...
  ✓ Cleaned 36 pages

[4/5] Adding metadata...
  ✓ Metadata added

[5/5] Chunking and storing...
  ✓ Stored 115 chunks in Pinecone

✓ Done! 'ethereum' ingested successfully.



# 7. Inference Phase (RAG)

In [10]:
def format_chat_history(history: list, max_turns: int = 5) -> str:
    """
    FORMAT CHAT HISTORY - Convert conversation history into a readable string.

    We limit to the last N turns to:
      - Stay within the LLM's context window
      - Keep costs low (fewer tokens = cheaper)
      - Focus on recent, relevant context

    Args:
        history (list): List of {role, content} message dicts
        max_turns (int): How many past turns to include

    Returns:
        str: Formatted conversation string
    """
    if not history:
        return "No previous conversation."

    # Each turn = 1 user message + 1 assistant message = 2 entries
    recent = history[-(max_turns * 2):]

    formatted = []
    for message in recent:
        role = message["role"].capitalize()
        content = message["content"]
        formatted.append(f"{role}: {content}")

    return "\n".join(formatted)

In [11]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

def create_rag_chain():
    """
    CREATE RAG CHAIN - Build the prompt + LLM pipeline.

    The chain takes three inputs:
      - context: retrieved document chunks
      - history: recent conversation turns
      - query: the user's current question

    Returns:
        RunnableSequence: An LCEL chain (prompt | llm | parser)
    """

    rag_template = ChatPromptTemplate.from_template(
        """
        You are a helpful assistant answering questions about cryptocurrency whitepapers.

        Use the conversation history and provided documents to answer the current question.
        If the question references previous context (e.g., "it", "this", "that"),
        use the conversation history to understand what is being referenced.

        Instructions:
        - Answer clearly and concisely (max 2-3 paragraphs)
        - Use only the provided documents as context
        - If the answer is not in the documents, say "I don't have enough information to answer that question"

        Conversation History:
        {history}

        Documents:
        {context}

        Current Question: {query}

        Answer:
        """
    )

    # LCEL chain: prompt → LLM → plain string
    rag_chain = rag_template | llm | StrOutputParser()

    return rag_chain

In [12]:
def ask(query: str, chat_history: list = None) -> str:
    """
    ASK - Run a single question through the RAG pipeline.

    Steps:
      1. Format chat history
      2. Retrieve the 3 most relevant chunks from Pinecone
      3. Pass chunks + history + query to the LLM
      4. Return the generated answer

    Args:
        query (str): The user's question
        chat_history (list): Previous conversation turns (optional)

    Returns:
        str: The LLM-generated answer
    """

    # Default to empty history if none provided
    if chat_history is None:
        chat_history = []

    # Step 1: Format history into a readable string
    formatted_history = format_chat_history(chat_history, max_turns=5)

    # Step 2: Retrieve top-3 relevant chunks using similarity search
    # k=3 is a good default: enough context without overwhelming the LLM
    results = vectorstore.similarity_search(query, k=3)

    # Step 3: Join chunks into a single context string
    context = "\n\n".join([doc.page_content for doc in results])

    # Step 4: Generate answer
    response = create_rag_chain().invoke({
        "context": context,
        "query": query,
        "history": formatted_history
    })

    return response

# 8. User Interface (Gradio)

Gradio creates an interactive chat interface directly inside Colab.
The UI handles conversation history automatically — no manual history management needed.

In [13]:
import gradio as gr

def gradio_chat(message: str, history: list) -> str:
    """
    GRADIO CHAT HANDLER

    Gradio calls this function every time the user sends a message.
    It receives the new message and the full conversation history,
    and must return the assistant's reply as a string.

    Gradio's history format is a list of [user, assistant] pairs:
      [["Hello", "Hi!"], ["What is Bitcoin?", "Bitcoin is..."]]

    We convert it to our internal format before passing it to ask().

    Args:
        message (str): The user's latest message
        history (list): Gradio conversation history (list of [user, assistant] pairs)

    Returns:
        str: The assistant's reply
    """

    # Convert Gradio's [user, assistant] pairs into our {role, content} format
    # so we can pass it to the existing ask() function unchanged
    chat_history = []
    for user_msg, assistant_msg in history:
        chat_history.append({"role": "user", "content": user_msg})
        chat_history.append({"role": "assistant", "content": assistant_msg})

    # Run the RAG pipeline with the converted history
    return ask(message, chat_history=chat_history)


# ── Build the Gradio interface ─────────────────────────────────
demo = gr.ChatInterface(
    fn=gradio_chat,                          # Function to call on each message
    title="Crypto Whitepapers Q&A",
    description="Ask questions about the Bitcoin and Ethereum whitepapers.",
    examples=[
        "What is Bitcoin?",
        "How does proof-of-work prevent double spending?",
        "What is Ethereum and how is it different from Bitcoin?",
        "What is the incentive for miners to support the network?"
    ],
    cache_examples=False   # Don't cache — answers depend on live Pinecone retrieval
)

# share=True creates a public link so you can share the demo outside Colab
demo.launch(share=True)

/usr/local/lib/python3.12/dist-packages/gradio/chat_interface.py:347: UserWarning: The 'tuples' format for chatbot messages is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style 'role' and 'content' keys.
  self.chatbot = Chatbot(


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://57dd29d5f27e03f030.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


# 9. Testing / Demo

In [14]:
# ── Single-turn questions ──────────────────────────────────────
# Test that the RAG pipeline retrieves relevant context and answers correctly.

questions = [
    "What is Bitcoin?",
    "How does proof-of-work prevent double spending?",
    "What is the purpose of the timestamp server in Bitcoin?",
    "What is Ethereum and how is it different from Bitcoin?",
    "What is the incentive for miners to support the Bitcoin network?"
]

for q in questions:
    print(f"Q: {q}")
    answer = ask(q)
    print(f"A: {answer}")
    print("-" * 60)

Q: What is Bitcoin?
A: Bitcoin is a decentralized peer-to-peer online currency introduced by Satoshi Nakamoto in January 2009. Unlike traditional currencies, it operates without any backing, intrinsic value, or central issuer. Bitcoin allows for direct online payments between parties without the need for a financial institution, addressing the double-spending problem through a peer-to-peer network that timestamps transactions using hash-based proof-of-work. This means that once a transaction is confirmed and added to the blockchain, it becomes part of an ongoing, immutable record.

From a technical perspective, Bitcoin can be viewed as a state transition system where the current state represents the ownership status of all existing bitcoins. Transactions serve as commands that modify this state, enabling a clear and verifiable mechanism for transferring ownership. The system ensures that only the first transaction of a given amount is processed, thus preventing double-spending and ensu

In [15]:
# ── Multi-turn conversation demo ──────────────────────────────
# Shows that the system can maintain context across follow-up questions.

# We build the history list manually to simulate a conversation
history = []

def chat(query: str):
    """Send a message and update the conversation history."""
    answer = ask(query, chat_history=history)

    # Append both sides of the turn to history
    history.append({"role": "user", "content": query})
    history.append({"role": "assistant", "content": answer})

    print(f"User: {query}")
    print(f"Assistant: {answer}")
    print("-" * 60)

# Simulate a conversation with follow-up questions
chat("What is Bitcoin?")
chat("How does it prevent double spending?")   # 'it' refers to Bitcoin — tests history
chat("Who gets the mining reward?")            # follow-up on the same topic

User: What is Bitcoin?
Assistant: Bitcoin is a decentralized peer-to-peer online currency that enables direct online payments between parties without the need for a financial institution or a central issuer. It was introduced by Satoshi Nakamoto in 2009 as part of a broader concept that includes a proof-of-work blockchain, which helps establish a public consensus on the order of transactions. The Bitcoin network resolves the double-spending problem by timestamping transactions and creating an immutable record through a changing chain of hashed proof-of-work. 

The Bitcoin ledger operates like a state transition system where each transaction modifies the ownership status of bitcoins. It ensures that transactions are valid based on the current state and prevents fraudulent activities, such as attempting to spend the same bitcoin more than once. Through this mechanism, Bitcoin establishes a reliable and secure method for value transfer, making it a revolutionary form of electronic cash.
-

# 10. RAGAS Evaluation

RAGAS (Retrieval Augmented Generation Assessment) measures RAG quality with four metrics:

- **Faithfulness** — is the answer grounded in the retrieved context, or is the LLM hallucinating?
- **Answer Relevancy** — does the answer actually address what was asked?
- **Context Precision** — are the most relevant chunks ranked at the top of the results?
- **Context Recall** — did we retrieve all the information needed to answer correctly?

All scores range from 0 to 1. Higher is better.

> **Cost note:** RAGAS makes LLM calls for each question × metric. With 6 questions this takes ~2-3 minutes and costs roughly $0.01-0.02.

In [16]:
# ── Evaluation Dataset ─────────────────────────────────────────
# Each item has a question and a ground_truth (the ideal answer).
# RAGAS uses the ground_truth to calculate Context Recall —
# it checks whether the retrieved chunks contain enough information
# to produce the correct answer.

EVALUATION_DATASET = [
    {
        "question": "What is Bitcoin?",
        "ground_truth": "Bitcoin is a decentralized digital currency that lets people transfer value "
                        "directly to one another over the internet, without relying on banks or other "
                        "central intermediaries. It uses a peer-to-peer network and a shared ledger "
                        "secured by cryptography and proof-of-work to record and order transactions."
    },
    {
        "question": "How does proof-of-work prevent double spending?",
        "ground_truth": "Proof-of-work makes altering transaction history extremely costly. Nodes accept "
                        "as valid the chain with the greatest accumulated work. For an attacker to spend "
                        "the same coins twice, they would need to rebuild the block containing the original "
                        "payment and all subsequent blocks, and then produce a longer chain than the honest "
                        "network, which becomes increasingly impractical as more blocks are added."
    },
    {
        "question": "What is the purpose of the timestamp server in Bitcoin?",
        "ground_truth": "The timestamp server gives transactions an ordered, verifiable place in history. "
                        "It groups data into blocks, hashes each block, and publishes the hash. Each new "
                        "block's hash includes the previous block's hash, creating a chain. This structure "
                        "proves that the recorded data existed at or before the time it was timestamped and "
                        "prevents past entries from being changed without redoing the work for all following blocks."
    },
    {
        "question": "How are transactions verified in the Bitcoin network?",
        "ground_truth": "Transactions are validated collectively by the network. Miners collect pending "
                        "transactions into a candidate block, check that each transaction follows the protocol "
                        "rules and does not reuse already spent outputs, and then perform proof-of-work on that block. "
                        "Once a miner finds a valid proof, it broadcasts the block; other full nodes accept it only if "
                        "the block itself and all included transactions are valid and unspent."
    },
    {
        "question": "What is the incentive for nodes to support the Bitcoin network?",
        "ground_truth": "Mining nodes are incentivized through economic rewards embedded in the protocol. "
                        "Each mined block contains a special transaction that creates new bitcoins for the block creator "
                        "(the block reward), and miners can also claim any difference between the total transaction inputs "
                        "and outputs as transaction fees. Together, these rewards motivate miners to spend resources securing the network."
    },
    {
        "question": "What is Ethereum and how is it different from Bitcoin?",
        "ground_truth": "Ethereum is a decentralized platform that runs smart contracts: applications that execute "
                        "exactly as programmed without any possibility of downtime, censorship, or fraud. "
                        "Unlike Bitcoin, which is primarily a digital currency and payment network, Ethereum is "
                        "a programmable blockchain that allows developers to build and deploy decentralized applications."
    },
]

print(f"Evaluation dataset ready: {len(EVALUATION_DATASET)} questions")

Evaluation dataset ready: 6 questions


In [17]:
import pandas as pd
from datasets import Dataset
from ragas import evaluate
from ragas.metrics import (
    faithfulness,
    answer_relevancy,
    context_precision,
    context_recall
)

def run_ragas_evaluation(dataset: list) -> pd.DataFrame:
    """
    RUN RAGAS EVALUATION

    Steps:
      1. Run each question through the RAG pipeline to collect answers + contexts
      2. Package everything into a RAGAS Dataset
      3. Score with four RAGAS metrics
      4. Return a summary DataFrame

    Args:
        dataset (list): List of {question, ground_truth} dicts

    Returns:
        pd.DataFrame: Per-question scores and overall averages
    """

    print("=" * 60)
    print("STEP 1/2 — Collecting answers and contexts...")
    print("=" * 60)

    questions = []
    answers = []
    contexts = []
    ground_truths = []

    for i, item in enumerate(dataset):
        question = item["question"]
        print(f"\n[{i+1}/{len(dataset)}] {question}")

        # Retrieve the top-3 chunks for this question
        # We collect them separately so RAGAS can evaluate retrieval quality
        results = vectorstore.similarity_search(question, k=3)
        retrieved_contexts = [doc.page_content for doc in results]
        context_str = "\n\n".join(retrieved_contexts)

        # Generate an answer using the RAG chain
        answer = create_rag_chain().invoke({
            "context": context_str,
            "query": question,
            "history": "No previous conversation."
        })

        questions.append(question)
        answers.append(answer)
        contexts.append(retrieved_contexts)   # RAGAS expects a list of strings per question
        ground_truths.append(item["ground_truth"])

        print(f"  ✓ Answer collected")

    print("\n" + "=" * 60)
    print("STEP 2/2 — Running RAGAS metrics...")
    print("  • Faithfulness")
    print("  • Answer Relevancy")
    print("  • Context Precision")
    print("  • Context Recall")
    print("=" * 60 + "\n")

    # Package into a HuggingFace Dataset — the format RAGAS expects
    ragas_dataset = Dataset.from_dict({
        "question": questions,
        "answer": answers,
        "contexts": contexts,
        "ground_truth": ground_truths
    })

    # Run all four metrics in one call
    # RAGAS uses the LLM and embeddings we already have configured
    scores = evaluate(
        ragas_dataset,
        metrics=[
            faithfulness,
            answer_relevancy,
            context_precision,
            context_recall
        ],
        embeddings=embeddings,
        llm=llm
    )

    # Convert to DataFrame for easy reading
    df = scores.to_pandas()

    # Print summary
    print("=" * 60)
    print("RESULTS")
    print("=" * 60)
    print(f"  Faithfulness:      {df['faithfulness'].mean():.3f}")
    print(f"  Answer Relevancy:  {df['answer_relevancy'].mean():.3f}")
    print(f"  Context Precision: {df['context_precision'].mean():.3f}")
    print(f"  Context Recall:    {df['context_recall'].mean():.3f}")
    print("=" * 60)

    return df

/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)
/tmp/ipykernel_6228/2571945355.py:4: DeprecationWarning: Importing faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import faithfulness
  from ragas.metrics import (
/tmp/ipykernel_6228/2571945355.py:4: DeprecationWarning: Importing answer_relevancy from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import answer_relevancy
  from ragas.metrics import

In [18]:
# Run the evaluation — this takes 2-3 minutes
results_df = run_ragas_evaluation(EVALUATION_DATASET)

# Add questions back — RAGAS drops them after to_pandas()
results_df.insert(0, "question", [item["question"] for item in EVALUATION_DATASET])

# Show per-question breakdown
# Useful for spotting which questions the system struggles with
print("\nPer-question breakdown:")
results_df[[
    "question",
    "faithfulness",
    "answer_relevancy",
    "context_precision",
    "context_recall"
]]

STEP 1/2 — Collecting answers and contexts...

[1/6] What is Bitcoin?
  ✓ Answer collected

[2/6] How does proof-of-work prevent double spending?
  ✓ Answer collected

[3/6] What is the purpose of the timestamp server in Bitcoin?
  ✓ Answer collected

[4/6] How are transactions verified in the Bitcoin network?
  ✓ Answer collected

[5/6] What is the incentive for nodes to support the Bitcoin network?
  ✓ Answer collected

[6/6] What is Ethereum and how is it different from Bitcoin?
  ✓ Answer collected

STEP 2/2 — Running RAGAS metrics...
  • Faithfulness
  • Answer Relevancy
  • Context Precision
  • Context Recall



Evaluating:   0%|          | 0/24 [00:00<?, ?it/s]

RESULTS
  Faithfulness:      0.967
  Answer Relevancy:  0.930
  Context Precision: 0.944
  Context Recall:    0.750

Per-question breakdown:


,question,faithfulness,answer_relevancy,context_precision,context_recall
0,What is Bitcoin?,0.866667,0.713597,1.000000,1.0
1,How does proof-of-work prevent double spending?,1.000000,0.941515,1.000000,1.0
2,What is the purpose of the timestamp server in...,1.000000,1.000000,1.000000,1.0
3,How are transactions verified in the Bitcoin n...,1.000000,1.000000,0.833333,1.0
4,What is the incentive for nodes to support the...,1.000000,0.948062,1.000000,0.5
5,What is Ethereum and how is it different from ...,0.937500,0.974137,0.833333,0.0
